In [1]:
import pydicom
import numpy as np
import pandas as pd

In [2]:
def applyWindow(image, center, width):
    image = image.copy()
    min_value = center - width // 2
    max_value = center + width // 2
    image[image < min_value] = min_value
    image[image > max_value] = max_value
    return image

In [3]:
def processImage(dicom):
    try:
        image = dicom.pixel_array
        slope = int(dicom['0028','1053'].value)
        intercept = int(dicom['0028','1052'].value)
        image = image * slope + intercept
        image1 = applyWindow(image, 40, 80)
        image2 = applyWindow(image, 80, 200)
        image3 = applyWindow(image, 40, 380)
        image1 = (image1 - 0) / 80
        image2 = (image2 - (-20)) / 200
        image3 = (image3 - (-150)) / 380
        image1 = image1 - image1.mean()
        image2 = image2 - image2.mean()
        image3 = image3 - image3.mean()
        image = np.array([image1,image2,image3]).transpose(1,2,0)
        image = image.astype(np.float16)
    except:
        print('error')
        image = np.zeros((512,512,3))
    return image

In [ ]:
def imageStats(name):
    try:
        image = pydicom.dcmread('../data/stage_1_test_images/{}.dcm'.format(name))
        slope, intercept = float(image.RescaleSlope), float(image.RescaleIntercept)
        image = image.pixel_array
        image = np.array(image * slope + intercept)
        w, l = 80,40
        return ((image > l-w//2) & (image < l+w//2)).mean().astype(np.float16)
    except:
        print('error')
        return 0.

In [4]:
train = pd.read_csv('../data/test.csv')
train['ID'] = train['ID'].map(lambda x : '_'.join(x.split('_')[:-1])) 
train = train['ID'].unique().tolist()

In [ ]:
for idx, name in enumerate(train):
    if idx % 10000 == 0:
        print(idx, 'images processed...')
    try:
        dicom = pydicom.dcmread('../data/test/{}.dcm'.format(name))
        image = processImage(dicom)
        np.savez_compressed('../output/test/{}.npz'.format(name), image)
    except:
        print('error')

0 images processed...
10000 images processed...
20000 images processed...
30000 images processed...
40000 images processed...
50000 images processed...
